In [16]:
import os
import mne
# mne.set_log_level("WARNING")
mne.set_log_level("ERROR")
import numpy as np
import pandas as pd
import pickle
import random
from sklearn.model_selection import train_test_split
from scipy.signal import detrend

In [2]:
import sys
print(sys.executable)
print(mne.__version__)
print(mne.__file__)

/Library/Developer/CommandLineTools/usr/bin/python3
1.8.0
/Users/neo/Library/Python/3.9/lib/python/site-packages/mne/__init__.py


In [3]:
# Paths for input csv folders
train_data_path = 'train'
chunk_size = 3.0
LINE_NOISE = 50.0

rs = 42 # seed number for data split
nchan = 56 # number of channels

In [4]:
labels_df = pd.read_csv(f'TrainLabels.csv')
labels_map = labels_df.set_index('IdFeedBack')['Prediction'].to_dict()
channels_loc = pd.read_csv(f'ChannelsLocation.csv')

output_folders = [f's{rs}_n{nchan}/train', f's{rs}_n{nchan}/val', f's{rs}_n{nchan}/test']
wired_file = []
# Create output folders if they don't exist
for folder in output_folders:
    os.makedirs(folder, exist_ok=True)

In [5]:
# Function to process csv files: downsample, chunk, and label
def process_csv(df, sfreq=200, selected_channels=None):
    ch_names = [col for col in df.columns if col not in ['Time', 'FeedBackEvent', 'EOG']]
    eeg_data = df[ch_names].T.values
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(eeg_data, info)

    feedback_events = df.index[df['FeedBackEvent'] == 1].tolist()

    if selected_channels:
        raw.pick_channels(selected_channels)

    if raw.info['sfreq'] != sfreq:
        feedback_events = [int(idx * sfreq / raw.info['sfreq']) for idx in feedback_events]

    signals = raw.get_data()
    return signals, feedback_events, ch_names

# Function to save chunks as pickle files
def save_epochs(epochs, folder):
    for i, (epoch_data, label, epoch_id) in enumerate(epochs):
        
        sample = {
            'signal': epoch_data,  # shape: (n_channels, 800)
            'label': label,
            'epoch_id': epoch_id
        }
        
        filename = os.path.join(folder, f"{epoch_id}.pickle")
        
        with open(filename, 'wb') as f:
            pickle.dump(sample, f)
        print(f"Saved: {filename}")


In [6]:
def prepare_and_slice_epochs(df, raw, epoch_duration, file_path, labels_map):
    feedback_events = df.index[df['FeedBackEvent'] == 1].tolist()
    epoch_samples = int(epoch_duration * raw.info['sfreq'])
    signals = raw.get_data()

    filename = os.path.basename(file_path)
    prefix = filename.replace('Data_', '').replace('.csv', '')

    epochs = []
    n_samples = signals.shape[1]

    for fb_idx, event_idx in enumerate(feedback_events):
        start_idx = event_idx
        end_idx = event_idx + epoch_samples

        if end_idx <= n_samples:
            epoch_data = signals[:, start_idx:end_idx].copy()
            epoch_id = f"{prefix}_FB{fb_idx + 1:03d}"
            label = labels_map.get(epoch_id, -1)
            if label != -1:
                epochs.append((epoch_data, label, epoch_id))
        else:
            print(end_idx, "<----超过了-------")

    return epochs, filename

## 滤波后切片

In [7]:
#读
def df_to_raw_full(df, sfreq=200):
    """从 DataFrame 生成整段 Raw"""
    ch_names = [c for c in df.columns if c not in ['Time', 'FeedBackEvent', 'EOG']]
    eeg_data = df[ch_names].T.values  # (n_ch, n_times)
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(eeg_data, info, verbose=False)
    return raw, ch_names

#滤波 简洁
def filter_full(raw, l_freq, h_freq, line_noise=50, target_sfreq=200):
    """对整段 raw 统一滤波→陷波→重采样。短段优先 IIR，长段 FIR。"""
    sf = raw.info['sfreq']
    raw.filter(l_freq, h_freq, n_jobs=-1)
    raw.notch_filter(line_noise, filter_length='auto', n_jobs=-1)
    if target_sfreq and target_sfreq != sf:
        raw.resample(target_sfreq, npad='auto', verbose=False)
    return raw

In [8]:
def remove_dc_offset(raw):
    """Remove DC offset per channel."""
    data, times = raw.get_data(return_times=True)
    data -= np.mean(data, axis=1, keepdims=True)
    raw._data = data
    return raw

def remove_linear_trend(raw):
    """Remove linear trend per channel."""
    try:
        data = detrend(raw.get_data(), axis=1, type='linear')
        # print("✅ detrend finished successfully")
    except Exception as e:
        print("❌ detrend failed:", e)
    raw._data = data
    return raw


def normalize_amplitude(raw, mode="div100"):
    """Amplitude normalization according to mode."""
    data = raw.get_data()
    if mode == "div100":
        raw._data = data / 100.0  # µV
    elif mode == "zscore_per_channel":
        mean = np.mean(data, axis=1, keepdims=True)
        std = np.std(data, axis=1, keepdims=True)
        raw._data = (data - mean) / std
    elif mode == "global_zscore":
        raw._data = (data - np.mean(data)) / np.std(data)
    elif mode == "percentile_95":
        p95 = np.percentile(np.abs(data), 95, axis=1, keepdims=True)
        raw._data = data / p95
    return raw
# def dc_offset(x):
#     """
#     Remove DC offset for each channel.
#     x: np.ndarray of shape (n_channels, n_samples)
#     """
#     return x - np.mean(x, axis=1, keepdims=True)

In [ ]:
def extract_epochs_cbramod(file_path, labels_map, sfreq=200, epoch_duration=chunk_size, line_noise=50):
    df = pd.read_csv(file_path)
    raw, ch_names = df_to_raw_full(df, sfreq=sfreq)
    raw = filter_full(raw, 0.3, 75.0, line_noise=line_noise, target_sfreq=200)
    # raw = normalize_amplitude(raw, "div100")#？代码里面有没有处以100
    epochs, filename = prepare_and_slice_epochs(df, raw, epoch_duration, file_path, labels_map)

    print(f"[CBraMod] extract {len(epochs)} epochs")
    if len(epochs) < 60:
        wired_file.append(filename)
    return epochs

# def LaBraM(raw, line_noise):
#     raw.filter(l_freq=0.1, h_freq=75.0, n_jobs=-1)
#     raw.notch_filter(line_noise, n_jobs=-1)
#     return raw
def extract_epochs_labram(file_path, labels_map, sfreq=200, epoch_duration=chunk_size, line_noise=50):
    df = pd.read_csv(file_path)
    raw, ch_names = df_to_raw_full(df, sfreq=sfreq)
    
    raw.resample(200)
    raw.filter(l_freq=0.1, h_freq=75.0, n_jobs=1)
    raw.notch_filter(line_noise, n_jobs=1)
    #代码内部做了除以100了
    epochs, filename = prepare_and_slice_epochs(df, raw, epoch_duration, file_path, labels_map)
    print(f"[LaBraM] extract {len(epochs)} epochs")
    if len(epochs) < 60:
        wired_file.append(filename)
    return epochs

def extract_epochs_neurolm(file_path, labels_map, sfreq=200, epoch_duration=chunk_size, line_noise=50):
    df = pd.read_csv(file_path)
    raw, ch_names = df_to_raw_full(df, sfreq=sfreq)

    raw.resample(200)
    raw.filter(l_freq=0.1, h_freq=75.0, n_jobs=1)
    raw.notch_filter(line_noise, n_jobs=1)
    # raw = normalize_amplitude(raw, "div100")#？代码有没有做div100

    epochs, filename = prepare_and_slice_epochs(df, raw, epoch_duration, file_path, labels_map)

    print(f"[NeuroLM] extract {len(epochs)} epochs")
    if len(epochs) < 60:
        wired_file.append(filename)
    return epochs

# def BIOT(raw, line_noise):
#     raw.filter(l_freq=0.1, h_freq=75.0, n_jobs=1) #？图中没有这一步，只有200hz和95
#     raw.notch_filter(line_noise, n_jobs=1)
#     raw.resample(200)
#     raw = normalize_amplitude(raw, "percentile_95")
#     return raw
def extract_epochs_biot(file_path, labels_map, sfreq=200, epoch_duration=chunk_size, line_noise=50):
    df = pd.read_csv(file_path)
    raw, ch_names = df_to_raw_full(df, sfreq=sfreq)
    # 严格模仿 BIOT(raw, line_noise)
    # raw.filter(l_freq=0.1, h_freq=75.0, n_jobs=1)
    # raw.notch_filter(line_noise, n_jobs=1)
    raw.resample(200)
    # raw = normalize_amplitude(raw, "percentile_95")#？代码有没有做percentile_95

    epochs, filename = prepare_and_slice_epochs(df, raw, epoch_duration, file_path, labels_map)
    print(f"[BIOT] extract {len(epochs)} epochs")
    if len(epochs) < 60:
        wired_file.append(filename)
    return epochs

# def EEGPT(raw, line_noise):
#     data = raw.get_data()
#     data -= np.mean(data, axis=1, keepdims=True)
#     raw._data = data
#     raw.resample(256, npad="auto", verbose='error')
#     raw.set_eeg_reference(ref_channels='average', projection=False)
#     return raw
def extract_epochs_eegpt(file_path, labels_map, sfreq=200, epoch_duration=chunk_size, line_noise=50):
    df = pd.read_csv(file_path)
    raw, ch_names = df_to_raw_full(df, sfreq=sfreq)

    # resample 256
    raw.resample(256, npad="auto", verbose='error')
    raw.filter(l_freq=0.1, h_freq=75.0, n_jobs=1)
    raw.notch_filter(line_noise, n_jobs=1)
    data = raw.get_data()
    raw = remove_dc_offset(raw)
    #global average reference
    raw.set_eeg_reference(ref_channels='average', projection=False)

    epochs, filename = prepare_and_slice_epochs(df, raw, epoch_duration, file_path, labels_map)

    print(f"[EEGPT] extract {len(epochs)} epochs")
    if len(epochs) < 60:
        wired_file.append(filename)
    return epochs

# def NeuroGPT(raw, line_noise):
#     bads = []
#     for ch_name in raw.ch_names:
#         data = raw.get_data(picks=ch_name)[0]
#         if np.allclose(data, 0) or np.isnan(data).all():
#             bads.append(ch_name)
#     raw.info['bads'] = bads
#     if len(raw.info['bads']) > 0:
#         raw.interpolate_bads(reset_bads=True, mode='accurate')
#     raw.filter(l_freq=0.5, h_freq=100.0, n_jobs=1)
#     raw.notch_filter(line_noise, n_jobs=1)
#     raw = dc_offset(raw)
#     raw = remove_linear_trend(raw)
#     raw.set_eeg_reference(ref_channels='average')
#     raw.resample(250)
#     raw = normalize_amplitude(raw, "zscore_per_channel")
#     return raw
def extract_epochs_neurogpt(file_path, labels_map, sfreq=200, epoch_duration=chunk_size, line_noise=50):
    df = pd.read_csv(file_path)
    raw, ch_names = df_to_raw_full(df, sfreq=sfreq)
    raw.resample(250, npad="auto", verbose='error')
    raw.filter(l_freq=0.5, h_freq=100.0, n_jobs=1)
    raw.notch_filter(line_noise, n_jobs=1)
    raw = remove_dc_offset(raw)
    raw = remove_linear_trend(raw)#这行会RuntimeWarning，小问题，不管
    raw.set_eeg_reference(ref_channels='average')

    # raw = normalize_amplitude(raw, "zscore_per_channel")#？代码有没有做zscore

    epochs, filename = prepare_and_slice_epochs(df, raw, epoch_duration, file_path, labels_map)

    print(f"[NeuroGPT] extract {len(epochs)} epochs")
    if len(epochs) < 60:
        wired_file.append(filename)
    return epochs

def extract_epochs_sttransformer(file_path, labels_map, sfreq=200, epoch_duration=chunk_size, line_noise=50):
    df = pd.read_csv(file_path)
    raw, ch_names = df_to_raw_full(df, sfreq=sfreq)
    raw.resample(250)
    raw.filter(l_freq=4.0, h_freq=40.0, n_jobs=1)
    raw.notch_filter(line_noise, n_jobs=1)
    # raw = normalize_amplitude(raw, "global_zscore") #？代码有没有做global zscore

    epochs, filename = prepare_and_slice_epochs(df, raw, epoch_duration, file_path, labels_map)

    print(f"[STTransformer] extract {len(epochs)} epochs")
    if len(epochs) < 60:
        wired_file.append(filename)
    return epochs

In [10]:
EXTRACTORS = {
    "cbramod": extract_epochs_cbramod,
    "labram": extract_epochs_labram,
    "biot": extract_epochs_biot,
    "eegpt": extract_epochs_eegpt,
    "neurolm": extract_epochs_neurolm,
    "neurogpt": extract_epochs_neurogpt,
    "sttransformer": extract_epochs_sttransformer,
}
def process_inria_bci_challenge(model_name="cbramod"):
    if model_name not in EXTRACTORS:
        raise ValueError(f"Unknown model_name: {model_name}. "
                         f"Choose from {list(EXTRACTORS.keys())}")
    extractor = EXTRACTORS[model_name]

    all_train_epochs = []
    train_files = sorted([f for f in os.listdir(train_data_path) if f.endswith('.csv')])

    for filename in train_files:
        file_path = os.path.join(train_data_path, filename)
        print(filename)
        # 只改这一行：按模型名选择对应的 extract 函数
        epochs = extractor(file_path, labels_map, 200, chunk_size, LINE_NOISE)
        all_train_epochs.extend(epochs)

    train_labels = [label for _, label, _ in all_train_epochs]
    print(f"Label distribution: {pd.Series(train_labels).value_counts().to_dict()}")

    train_epochs, temp_epochs = train_test_split(
        all_train_epochs,
        test_size=0.2,
        random_state=rs,
        stratify=train_labels,
        shuffle=True
    )

    temp_labels = [label for _, label, _ in temp_epochs]
    val_epochs, test_epochs = train_test_split(
        temp_epochs,
        test_size=0.5,
        random_state=rs,
        stratify=temp_labels,
        shuffle=True
    )

    total = len(all_train_epochs)
    print(f"Train: {len(train_epochs)}/{total} ({len(train_epochs)/total:.1%})")
    print(f"Val: {len(val_epochs)}/{total} ({len(val_epochs)/total:.1%})")
    print(f"Test: {len(test_epochs)}/{total} ({len(test_epochs)/total:.1%})")

    random.seed(rs)
    random.shuffle(train_epochs)
    random.shuffle(val_epochs)
    random.shuffle(test_epochs)

    save_epochs(train_epochs, f's{rs}_n{nchan}/train')
    save_epochs(val_epochs, f's{rs}_n{nchan}/val')
    save_epochs(test_epochs, f's{rs}_n{nchan}/test')


In [17]:
model_name="neurogpt"  # 可选 "cbramod", "labram", "biot", "eegpt", "neurolm", "neurogpt", "sttransformer"
process_inria_bci_challenge(model_name)

print("All files processed, split, and saved successfully!")
print(wired_file)

Data_S02_Sess01.csv


/Users/neo/Library/Python/3.9/lib/python/site-packages/scipy/signal/_signaltools.py:3602: RuntimeWarning: divide by zero encountered in matmul
  newdata[sl] = newdata[sl] - A @ coef
/Users/neo/Library/Python/3.9/lib/python/site-packages/scipy/signal/_signaltools.py:3602: RuntimeWarning: overflow encountered in matmul
  newdata[sl] = newdata[sl] - A @ coef
/Users/neo/Library/Python/3.9/lib/python/site-packages/scipy/signal/_signaltools.py:3602: RuntimeWarning: invalid value encountered in matmul
  newdata[sl] = newdata[sl] - A @ coef


[NeuroGPT] extract 60 epochs
Data_S02_Sess02.csv


/Users/neo/Library/Python/3.9/lib/python/site-packages/scipy/signal/_signaltools.py:3602: RuntimeWarning: divide by zero encountered in matmul
  newdata[sl] = newdata[sl] - A @ coef
/Users/neo/Library/Python/3.9/lib/python/site-packages/scipy/signal/_signaltools.py:3602: RuntimeWarning: overflow encountered in matmul
  newdata[sl] = newdata[sl] - A @ coef
/Users/neo/Library/Python/3.9/lib/python/site-packages/scipy/signal/_signaltools.py:3602: RuntimeWarning: invalid value encountered in matmul
  newdata[sl] = newdata[sl] - A @ coef


[NeuroGPT] extract 60 epochs
Label distribution: {1: 93, 0: 27}
Train: 96/120 (80.0%)
Val: 12/120 (10.0%)
Test: 12/120 (10.0%)
Saved: s42_n56/train/S02_Sess01_FB040.pickle
Saved: s42_n56/train/S02_Sess01_FB060.pickle
Saved: s42_n56/train/S02_Sess02_FB030.pickle
Saved: s42_n56/train/S02_Sess02_FB041.pickle
Saved: s42_n56/train/S02_Sess01_FB007.pickle
Saved: s42_n56/train/S02_Sess02_FB009.pickle
Saved: s42_n56/train/S02_Sess02_FB053.pickle
Saved: s42_n56/train/S02_Sess02_FB012.pickle
Saved: s42_n56/train/S02_Sess02_FB052.pickle
Saved: s42_n56/train/S02_Sess02_FB010.pickle
Saved: s42_n56/train/S02_Sess01_FB050.pickle
Saved: s42_n56/train/S02_Sess02_FB060.pickle
Saved: s42_n56/train/S02_Sess01_FB019.pickle
Saved: s42_n56/train/S02_Sess02_FB015.pickle
Saved: s42_n56/train/S02_Sess01_FB031.pickle
Saved: s42_n56/train/S02_Sess02_FB039.pickle
Saved: s42_n56/train/S02_Sess02_FB002.pickle
Saved: s42_n56/train/S02_Sess02_FB022.pickle
Saved: s42_n56/train/S02_Sess01_FB008.pickle
Saved: s42_n56/tra